# Supplier Product Enrichment & Sourcing Pipeline

This notebook drives the end-to-end pipeline for enriching raw supplier product pages into structured data and then using that data to source raw material components.

**Pipeline stages:**
1. **Extraction** — Gemini parses each supplier's HTML product page into structured fields (prices, purity, quality score, compliance metrics). Run once per supplier batch; results persist in SQLite.
2. **Export / Import** — Snapshot extracted data to `data/extracted_products.json` so collaborators can load without re-running Gemini.
3. **Constraint filtering** — `filter_products.make_filters()` builds a composable filter list from price, quantity, purity, quality-score, and per-metric compliance ranges.
4. **Supplier assignment (batching)** — `db.batch()` uses greedy set-cover to assign each BOM component to the fewest possible suppliers, optionally constrained by filters.
5. **Ranking** — `filter_products.rank_suppliers()` wraps batch + scoring into a single call that returns assignments sorted by composite quality score.
6. **Evaluation** — `evaluate.evaluate_batch()` samples BOMs and uses Gemini-as-judge (DeepEval GEval) to score each supplier–component assignment against the supplier's scraped HTML page.

**Key files:**
| File | Purpose |
|------|---------|
| `scripts/db.py` | SQLite helpers: upsert, query, `batch()` set-cover |
| `scripts/text2product.py` | Gemini extraction prompt + `COMPLIANCE_METRICS` |
| `scripts/filter_products.py` | Filters, `make_filters`, `score_product`, `rank_suppliers` |
| `scripts/evaluate.py` | DeepEval GEval evaluation harness |
| `data/supplier_products/` | Scraped HTML pages (one file per supplier × product) |
| `data/extracted_products.json` | Cached extraction results |


## 1. Extract product information from HTML pages

Loops over every HTML file in `data/supplier_products/`, calls `text2product.extract_product_info()` (Gemini), and writes results to SQLite via `db.upsert_supplier_product_*`.

**Filename convention:** `{supplier_id}_{supplier_name}_{product_id}_{sku_slug}.html`

Set `FORCE = True` to reprocess files that have already been extracted. By default the cell skips already-processed `(supplier_id, product_id)` pairs.

Extracted fields per product:
- `prices` — list of `{quantity, unit, price, currency}` tier dicts
- `purity` — fractional purity (0–1), e.g. 0.99
- `quality` — free-text quality description
- `quality_score` — 0–1 numeric grade (0.7 = food/kosher, 0.9 = GMP, 1.0 = USP/pharma)
- `compliance` — dict of metric → value (see `COMPLIANCE_METRICS` in `text2product.py`)


In [1]:
import importlib
import re
import sys
sys.path.insert(0, ".")

import db, text2product
importlib.reload(text2product)
importlib.reload(db)

from pathlib import Path
from text2product import html_to_text, extract_product_info
from db import upsert_supplier_product_prices, upsert_supplier_product_info, get_processed_supplier_products

SUPPLIER_PRODUCTS_DIR = Path("../data/supplier_products")
FNAME_RE = re.compile(r"^(\d+)_.+?_(\d+)_(RM-[^.]+)\.html$")
FORCE = False  # set True to reprocess everything from scratch

already_done = set() if FORCE else get_processed_supplier_products()
files = sorted(SUPPLIER_PRODUCTS_DIR.glob("*.html"))
print(f"Processing {len(files)} files ({len(already_done)} already done)...")

for path in files:
    m = FNAME_RE.match(path.name)
    if not m:
        print(f"  SKIP (unrecognised filename): {path.name}")
        continue

    supplier_id, product_id, sku_slug = int(m.group(1)), int(m.group(2)), m.group(3)

    if (supplier_id, product_id) in already_done:
        continue

    try:
        page_text = html_to_text(path.read_text(errors="replace"))
        result = extract_product_info(page_text, sku_slug)
    except Exception as e:
        print(f"  ERROR {path.name}: {e}")
        continue

    prices        = result.get("prices") or []
    purity        = result.get("purity")
    quality       = result.get("quality")
    quality_score = result.get("quality_score")
    compliance    = result.get("compliance") or {}

    if prices:
        upsert_supplier_product_prices(supplier_id, product_id, prices)
    upsert_supplier_product_info(supplier_id, product_id, purity, quality, quality_score, compliance)
    already_done.add((supplier_id, product_id))

    metrics_present = [k for k, v in compliance.items() if v is not None]
    print(f"  {path.name}: {len(prices)} tier(s), purity={purity}, quality={quality!r}, metrics={metrics_present}")

print("Done.")


Processing 203 files (1 already done)...
  24_Nutra_Blend_282_RM-C13-energy-support-botanicals-nutrients-4823efcb.html: 1 tier(s), purity=None, quality='feed grade', metrics=[]
  24_Nutra_Blend_582_RM-C33-ferment-media-c911f8c0.html: 1 tier(s), purity=None, quality='feed grade', metrics=[]
  24_Nutra_Blend_751_RM-C45-prebiotic-fiber-19375eea.html: 1 tier(s), purity=None, quality='feed grade', metrics=[]
  24_Nutra_Blend_820_RM-C52-performance-support-nutrients-165afec4.html: 1 tier(s), purity=0.84, quality='feed grade', metrics=['assay_potency']
  24_Nutra_Blend_885_RM-C57-cultured-nutrients-5c87fee4.html: 1 tier(s), purity=None, quality='feed grade', metrics=[]
  24_Nutra_Blend_886_RM-C57-fruit-nutrients-b65036d4.html: 1 tier(s), purity=None, quality='feed grade', metrics=[]
  24_Nutra_Blend_906_RM-C57-vegetable-nutrients-4667fd2d.html: 1 tier(s), purity=None, quality=None, metrics=[]
  24_Nutra_Blend_963_RM-C60-organic-food-complex-83c82fa3.html: 1 tier(s), purity=None, quality='feed

## 2. Export / Import extracted data

**Export** — snapshots the current DB state to `data/extracted_products.json`. Run after extraction to share results without requiring collaborators to have a Gemini API key.

**Import** — loads `extracted_products.json` into a fresh DB. Run this instead of the Gemini extraction cell if you just want to work with pre-extracted data.


In [2]:
# Export extracted data to JSON so others can import without re-running Gemini
import importlib, json, sys
sys.path.insert(0, ".")
import db; importlib.reload(db)
from db import get_supplier_products_enriched
from pathlib import Path

out = Path("../data/extracted_products.json")
data = get_supplier_products_enriched()
# only export rows that have been processed
data = [d for d in data if d.get("purity") is not None or d.get("quality") is not None or any(d.get("compliance", {}).values()) or d.get("prices")]
out.write_text(json.dumps(data, indent=2))
print(f"Exported {len(data)} processed rows to {out}")


Exported 167 processed rows to ../data/extracted_products.json


In [3]:
# Import extracted data into a fresh DB (run this instead of the Gemini extraction cell)
import importlib, json, sys
sys.path.insert(0, ".")
import db; importlib.reload(db)
from db import upsert_supplier_product_prices, upsert_supplier_product_info

data = json.loads(Path("../data/extracted_products.json").read_text())
for d in data:
    if d.get("prices"):
        upsert_supplier_product_prices(d["supplier_id"], d["product_id"], d["prices"])
    upsert_supplier_product_info(
        d["supplier_id"], d["product_id"],
        d.get("purity"), d.get("quality"), d.get("quality_score"), d.get("compliance"),
    )
print(f"Imported {len(data)} rows.")


Imported 167 rows.


## 3. Constraint filtering

`make_filters()` returns a list of callables that can be passed to `filter_products()` or directly to `db.batch()`.

Each range is `(lo, hi)` — either bound can be `None` for unbounded. Products missing a field pass through by default (opt-in filtering).

**Available ranges:**
- `price_range` — per-unit price in any tier
- `quantity_range` — minimum order quantity in any tier
- `purity_range` — fractional purity (0–1)
- `quality_range` — composite quality score (0–1); ~0.7 food grade, ~0.9 GMP, 1.0 USP/pharma
- `quality_metrics` — dict of `{metric_name: (lo, hi)}` for individual compliance metrics

All metrics from `COMPLIANCE_METRICS` also have dedicated filter helpers (e.g. `heavy_metals_filter(hi=10.0)`).


In [10]:
import importlib
import sys
sys.path.insert(0, ".")

import db, text2product, filter_products
importlib.reload(text2product)
importlib.reload(filter_products)
importlib.reload(db)

from db import get_supplier_products_enriched
from filter_products import filter_products as apply_filters, make_filters

price_range    = (None, None)
quantity_range = (None, None)
purity_range   = (None, None)
quality_range  = (None, None)  # 0-1: 0.7=food/kosher, 0.9=GMP, 1.0=USP/pharma
quality_metrics = {
    # "identity_confidence": (0.95, None),
    # "assay_potency":       (0.97, 1.03),
    # "heavy_metals":        (None, 10.0),
    # "pesticide_residues":  (None, 100.0),
    # "microbial_limits":    (1.0,  None),
    # "moisture_content":    (None, 0.05),
    # "residual_solvents":   (None, 410.0),
}

products = get_supplier_products_enriched()
filters  = make_filters(price_range, quantity_range, purity_range, quality_range, quality_metrics)
results  = apply_filters(products, filters)

print(f"{len(results)}/{len(products)} products passed")
results


1633/1633 products passed


[{'supplier_id': 1,
  'product_id': 182,
  'supplier_name': 'ADM',
  'sku': 'RM-C2-soy-lecithin-cc38c49d',
  'purity': None,
  'quality': None,
  'quality_score': None,
  'compliance': {},
  'prices': []},
 {'supplier_id': 1,
  'product_id': 202,
  'supplier_name': 'ADM',
  'sku': 'RM-C5-medium-chain-triglycerides-mct-from-coconut-oil-69d4233c',
  'purity': None,
  'quality': None,
  'quality_score': None,
  'compliance': {},
  'prices': []},
 {'supplier_id': 1,
  'product_id': 215,
  'supplier_name': 'ADM',
  'sku': 'RM-C6-soy-lecithin-5de90202',
  'purity': None,
  'quality': None,
  'quality_score': None,
  'compliance': {},
  'prices': []},
 {'supplier_id': 1,
  'product_id': 217,
  'supplier_name': 'ADM',
  'sku': 'RM-C6-sunflower-lecithin-47e33a0e',
  'purity': None,
  'quality': None,
  'quality_score': None,
  'compliance': {},
  'prices': []},
 {'supplier_id': 1,
  'product_id': 232,
  'supplier_name': 'ADM',
  'sku': 'RM-C8-sunflower-lecithin-f6a00a49',
  'purity': None,
  'q

### Example — filtered product search

Demonstrates chaining multiple constraints. Adjust any range or comment out lines to relax constraints.


In [11]:
products = get_supplier_products_enriched()

filters = make_filters(
    price_range=(None, 50.0),
    quantity_range=(100.0, None),
    purity_range=(0.98, None),
    quality_range=(0.9, None),
    quality_metrics={
        "identity_confidence": (0.95, None),
        "assay_potency":       (0.97, 1.03),
        "heavy_metals":        (None, 10.0),
        "microbial_limits":    (1.0,  None),
    },
)

results = apply_filters(products, filters)
print(f"{len(results)}/{len(products)} products passed")
results


1578/1633 products passed


[{'supplier_id': 1,
  'product_id': 182,
  'supplier_name': 'ADM',
  'sku': 'RM-C2-soy-lecithin-cc38c49d',
  'purity': None,
  'quality': None,
  'quality_score': None,
  'compliance': {},
  'prices': []},
 {'supplier_id': 1,
  'product_id': 202,
  'supplier_name': 'ADM',
  'sku': 'RM-C5-medium-chain-triglycerides-mct-from-coconut-oil-69d4233c',
  'purity': None,
  'quality': None,
  'quality_score': None,
  'compliance': {},
  'prices': []},
 {'supplier_id': 1,
  'product_id': 215,
  'supplier_name': 'ADM',
  'sku': 'RM-C6-soy-lecithin-5de90202',
  'purity': None,
  'quality': None,
  'quality_score': None,
  'compliance': {},
  'prices': []},
 {'supplier_id': 1,
  'product_id': 217,
  'supplier_name': 'ADM',
  'sku': 'RM-C6-sunflower-lecithin-47e33a0e',
  'purity': None,
  'quality': None,
  'quality_score': None,
  'compliance': {},
  'prices': []},
 {'supplier_id': 1,
  'product_id': 232,
  'supplier_name': 'ADM',
  'sku': 'RM-C8-sunflower-lecithin-f6a00a49',
  'purity': None,
  'q

## 4. Supplier ranking

`rank_suppliers(finished_good_sku)` combines batching + scoring in one call:
1. Calls `db.batch()` (greedy set-cover) to assign each BOM component to the fewest suppliers.
2. Scores each assignment with `score_product()` — a weighted average of `quality_score`, `purity`, `identity_confidence`, and `assay_potency` (deviation-penalised).
3. Returns `{suppliers, assignments, uncovered, ranked}` where `ranked` is sorted by score descending.

Pass `filters=make_filters(...)` to restrict eligible suppliers before the set-cover step.
Pass `weights={...}` to override the default scoring weights.


In [1]:
import importlib
import sys
sys.path.insert(0, ".")

import db, filter_products
importlib.reload(db)
importlib.reload(filter_products)

from db import get_boms
from filter_products import make_filters, rank_suppliers, score_product


# ── Example: pick first available finished-good SKU from the DB ──────────────
boms = get_boms()
if not boms:
    print("No BOMs found in DB.")
else:
    example_sku = boms[0]["ProducedSKU"]
    result = rank_suppliers(example_sku)

    print(f"Finished good : {example_sku}")
    print(f"Suppliers chosen: {result['suppliers']}")
    print(f"Uncovered SKUs  : {result['uncovered']}")
    print()

    if result["ranked"]:
        top = result["ranked"][0]
        print(f"── Top assignment ──────────────────────────────")
        print(f"  Supplier  : {top['supplier']}")
        print(f"  SKU       : {top['sku']}")
        print(f"  Score     : {top['score']}")
        print(f"  Purity    : {top['purity']}")
        print(f"  Quality   : {top['quality']}  (quality_score={top['quality_score']})")
        print(f"  Compliance metrics:")
        for k, v in top["compliance"].items():
            if v is not None:
                print(f"    {k:25s}: {v}")
        print(f"  Price tiers:")
        for tier in top["prices"]:
            qty = f"{tier['quantity']} {tier['unit']}" if tier.get("quantity") else "—"
            print(f"    {qty:20s} @ {tier['price']} {tier['currency']}")


Finished good : FG-amazon-b0002wrqy4
Suppliers chosen: ['Prinova USA', 'AIDP', 'American Botanicals', 'ADM', 'Ashland', 'Cambrex', 'Ingredion', 'Nutri Avenue']
Uncovered SKUs  : []

── Top assignment ──────────────────────────────
  Supplier  : Prinova USA
  SKU       : RM-C2-pyridoxine-hcl-0a843bcb
  Score     : 0.0
  Purity    : None
  Quality   : None  (quality_score=None)
  Compliance metrics:
  Price tiers:


## 5. Evaluation

`evaluate_batch()` samples BOMs, runs `batch()` on each, and scores every supplier–component assignment using Gemini-as-judge (DeepEval GEval).

The judge reads the supplier's scraped HTML page and scores 0–1 whether it confirms the supplier sells the specified raw material. Assignments with no HTML are skipped.

**To get better HTML coverage first:** run the CSV-direct scraping cell in `scrape_supplier_info.ipynb` — it fetches each URL from `data/fully_sourced_products_with_links.csv` directly (no fuzzy catalog crawl), then re-run the extraction cell above to populate prices/quality.

In [3]:
import importlib, sys
sys.path.insert(0, ".")

import evaluate
importlib.reload(evaluate)

from evaluate import evaluate_batch
from filter_products import make_filters

filters = make_filters(
    # price_range=(None, 200.0),
    # purity_range=(0.95, None),
    # quality_range=(0.7, None),
)

results = evaluate_batch(n_samples=10, filters=filters or None, seed=42)
results



--- FG-cvs-448437 ---
  RM-C37-copper-sulfate-31751952 → PureBulk: no HTML, skipping
  RM-C37-pyridoxine-hydrochloride-67345b25 → PureBulk: no HTML, skipping
  RM-C37-d-calcium-pantothenate-baa38478 → PureBulk: no HTML, skipping
  RM-C37-cholecalciferol-f6d103e4 → PureBulk: no HTML, skipping
  RM-C37-niacinamide-51ff204b → PureBulk: no HTML, skipping
  RM-C37-ascorbic-acid-2dce8c2c → PureBulk: no HTML, skipping
  RM-C37-riboflavin-ece92ffe → PureBulk: no HTML, skipping
  RM-C37-chromium-chloride-29a65259 → PureBulk: no HTML, skipping
  RM-C37-thiamine-mononitrate-1795539a → PureBulk: no HTML, skipping
  RM-C37-biotin-2d59ee9a → PureBulk: no HTML, skipping
  RM-C37-vitamin-a-acetate-c9020dfa → PureBulk: no HTML, skipping
  RM-C37-folic-acid-2bb2a34c → PureBulk: no HTML, skipping
  RM-C37-manganese-sulfate-1324ef0a → PureBulk: no HTML, skipping
  RM-C37-zinc-oxide-b733fd8f → PureBulk: no HTML, skipping
  RM-C37-dl-alpha-tocopheryl-acetate-50d2f8bd → PureBulk: no HTML, skipping
  RM-C37-

  RM-C37-gelatin-a40fdc3f → Capsuline: 0.60 (pass) — The Retrieval Context confirms Capsuline sells gelatin capsules suitable for supplement manufacturing, aligning with the component type and finished-good SKU implied by FG-cvs-448437, specifically noting '100% bovine hide' and 'Kosher and Halal certified'. However, it does not explicitly confirm the exact raw material component SKU RM-C37-gelatin-a40fdc3f. Furthermore, the Actual Output fails to include any pricing, preventing the evaluation of alignment with the price information of €362,95 found in the Retrieval Context, which significantly impacts its overall utility as per the evaluation steps.
  RM-C37-magnesium-oxide-f07b6a9a → Jost Chemical: no HTML, skipping
  mean: 0.60

--- FG-amazon-b07z2x2xtc ---
  RM-C4-calcium-c77f1de7 → Jost Chemical: no HTML, skipping
  RM-C4-magnesium-7058dcb8 → Jost Chemical: no HTML, skipping
  RM-C4-potassium-4fd2e03c → Jost Chemical: no HTML, skipping
  mean: n/a

--- FG-target-a-91641237 ---
  R

  RM-C14-gelatin-017eb28d → Capsuline: 0.60 (pass) — The response correctly identifies 'Capsuline' as the supplier, which is explicitly confirmed by the Retrieval Context selling 'Clear Gelatin Capsules Size 0' for the specified component SKU (aligning well with step 1). The Retrieval Context also clearly indicates the raw material's suitability for supplement manufacturing with details on purity, quality, and certifications (addressing the core intent of step 2). However, the Actual Output does not detail the purity and quality or provide any prices, thereby failing to fully align with step 2 and completely failing to align with step 3's explicit requirement to evaluate prices provided in the Actual Output.
  mean: 0.60

--- FG-gnc-145223 ---
  RM-C19-hydroxypropylmethylcellulose-eb1c22cf → Ashland: no HTML, skipping
  RM-C19-silicon-dioxide-8f3934a0 → Ashland: no HTML, skipping
  RM-C19-polysorbate-80-d1fa702e → Ashland: no HTML, skipping
  RM-C19-magnesium-stearate-63974975 → Ashlan

[{'produced_sku': 'FG-cvs-448437',
  'assignments_judged': [{'component_sku': 'RM-C37-copper-sulfate-31751952',
    'supplier': 'PureBulk',
    'score': None,
    'passed': None,
    'reason': 'no HTML'},
   {'component_sku': 'RM-C37-pyridoxine-hydrochloride-67345b25',
    'supplier': 'PureBulk',
    'score': None,
    'passed': None,
    'reason': 'no HTML'},
   {'component_sku': 'RM-C37-d-calcium-pantothenate-baa38478',
    'supplier': 'PureBulk',
    'score': None,
    'passed': None,
    'reason': 'no HTML'},
   {'component_sku': 'RM-C37-cholecalciferol-f6d103e4',
    'supplier': 'PureBulk',
    'score': None,
    'passed': None,
    'reason': 'no HTML'},
   {'component_sku': 'RM-C37-niacinamide-51ff204b',
    'supplier': 'PureBulk',
    'score': None,
    'passed': None,
    'reason': 'no HTML'},
   {'component_sku': 'RM-C37-ascorbic-acid-2dce8c2c',
    'supplier': 'PureBulk',
    'score': None,
    'passed': None,
    'reason': 'no HTML'},
   {'component_sku': 'RM-C37-riboflavin-